# pgvector and jev: AI inside Postgres

Two Postgres extensions in the `shop` database:

* **pgvector**: a `vector` column type, distance operators (`<=>` cosine, `<->` L2,
  `<#>` inner product) and HNSW indexes. Spark writes the vectors (job 18).
* **jev** ([realZachi/pg-jev](https://github.com/realZachi/pg-jev)): plain-language
  conditions in SQL. `jev()`, `jev_prob()`, `jev_choice()`, `jev_score()` judge each row
  through a Jev-compatible API.

By default jev talks to the lab's **jev-local** service, an offline stand-in that matches
words against a small lexicon. It is not a model, so expect rough answers. For real
judgments set `JEV_API_URL` and `TYPESAFE_API_KEY` in `.env` and run `make ai-reset-jev`.
The embeddings are the lab's hashing-trick `lab_embed()`, also not a model: texts that
share words end up close.

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

## Is everything installed?

In [ ]:
lab.pg("SELECT extname, extversion FROM pg_extension WHERE extname IN ('vector', 'jev', 'plpython3u') ORDER BY 1")

In [ ]:
lab.pg("SELECT jev_version() AS jev, current_setting('jev.api_url') AS api_url")

## jev: a WHERE clause in plain language

In [ ]:
lab.pg("SELECT * FROM people WHERE jev(people, 'the name is European')")

Cheap predicates run first, so jev only judges the rows that survive them:

In [ ]:
lab.pg("SELECT name, country, age FROM people WHERE age > 40 AND jev(people, 'works in healthcare') ORDER BY age DESC")

## jev_prob: rank by probability

In [ ]:
angry = lab.pg("""SELECT subject, jev_prob(tickets, 'the customer is angry') AS p
                   FROM tickets ORDER BY p DESC LIMIT 20""")
angry.head(10)

In [ ]:
angry.set_index('subject')['p'].astype(float).sort_values().plot.barh(figsize=(7, 6), title='P(customer is angry)');

## jev_choice: route every ticket

In [ ]:
teams = lab.pg("""SELECT jev_choice(tickets, 'which team should handle this?',
                                    ARRAY['billing', 'technical', 'security', 'sales']) AS team, count(*)
                   FROM tickets GROUP BY 1 ORDER BY 2 DESC""")
teams

## jev_score: position on ordered levels (0 = budget, 3 = luxury)

In [ ]:
lux = lab.pg("""SELECT name, price, jev_score(products, 'how luxurious is this product?',
                                  ARRAY['budget', 'mid-range', 'premium', 'luxury']) AS luxury
                 FROM products ORDER BY luxury DESC""")
lux.head(10)

In [ ]:
lux.astype({'price': float, 'luxury': float}).plot.scatter(x='price', y='luxury', logx=True, title='price vs jev luxury score');

## pgvector: Spark writes the embeddings
Job 18 embeds every product in Spark, upserts the vectors, and checks they equal Postgres' own `lab_embed()`.

In [ ]:
r = lab.run('pgvector-embeddings', run_id=RID)
lab.log_tail(r, 'search')

In [ ]:
def search(q, k=5):
    return lab.pg("""SELECT p.name, p.price, (e.embedding <=> lab_embed(%s)) AS distance
                     FROM product_embeddings e JOIN products p ON p.id = e.product_id
                     ORDER BY e.embedding <=> lab_embed(%s) LIMIT %s""", (q, q, k))
search("warm clothes for cold winter")

In [ ]:
search("professional kitchen knife")

Help articles were embedded in SQL by `pg-init`. Same function, same space:

In [ ]:
lab.pg("""SELECT title, embedding <=> lab_embed(%s) AS distance
           FROM kb_articles ORDER BY 2 LIMIT 3""", ("suspicious login alert",))

## Both together: route tickets and suggest an article, then land it in the lake
Job 19 sends one query to Postgres (jev + pgvector per row) and writes the result to Parquet, a Hive table and back to Postgres.

In [ ]:
r = lab.run('jev-tickets-to-lake', run_id=RID)
lab.log_tail(r, 'tickets')

In [ ]:
lab.sql("SELECT team, count(*) AS tickets, round(avg(angry_p), 3) AS avg_angry FROM ai.ticket_routing GROUP BY team ORDER BY tickets DESC")

In [ ]:
lab.pg("""SELECT r.ticket_id, t.subject, r.team, round(r.angry_p::numeric, 2) AS angry, k.title AS article
           FROM ticket_routing r JOIN tickets t ON t.id = r.ticket_id
           LEFT JOIN kb_articles k ON k.id = r.kb_article
           ORDER BY r.angry_p DESC LIMIT 10""")

## What did jev cost this session?
`jev_stats()` is per Postgres session; `lab.pg()` opens a new one each call, so this shows one query's numbers.

In [ ]:
lab.pg("SELECT jev_stats() AS stats, (SELECT count(*) FROM tickets WHERE jev(tickets, 'the customer is angry')) AS angry_tickets")